# Settings

## Import Packages

In [1]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import requests
import time
from geopy.geocoders import Nominatim
import time
from shapely.wkt import dumps
from shapely.wkt import loads

pd.set_option('display.precision', 4)

## Self-defined functions

### check_table_info

In [2]:
def check_table_info(target_df):
    """
    To check the unique values, dtype, example, missing rate of each column
    """
    table_info = []
    for col in target_df:
        table_info_row = []
        table_info_row.append(col)
        table_info_row.append(target_df[col].nunique())
        table_info_row.append(target_df[col].dtype)
        table_info_row.append(target_df[col].iloc[0])
        table_info_row.append(round(target_df[col].isna().sum() / target_df.shape[0]*100,2))

        table_info.append(table_info_row)
    res = pd.DataFrame(table_info, columns=['col_name', 'unique_values', 'dtype', 'example', 'missing%'])

    return res

### load_geoDataFrame

In [3]:
def load_geoDataFrame(filepath):
    """
    Load a GeoDataFrame from a CSV file with a geometry column in WKT format.

    Parameters:
        filepath (str): Path to the CSV file containing the saved GeoDataFrame.

    Returns:
        GeoDataFrame: A GeoDataFrame reconstructed from the CSV, with geometries parsed from WKT strings.
    
    Notes:
        - Assumes the geometry column is named 'geometry' and stored in WKT format.
        - The returned GeoDataFrame is assigned the CRS 'EPSG:4326' by default.
    """
    df = pd.read_csv(filepath)
    df['geometry'] = df['geometry'].apply(loads)
    return gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

### gen_valid_trip_influence_df

In [4]:
def gen_valid_trip_influence_df(
    trip_df, start_time_col, end_time_col, valid_duration_hour,
    start_location_col, end_location_col, trip_id_col=None, location_ids=None
):
    """
    Generate time-location influence DataFrame from trip data with ±1 hour influence period.

    Parameters:
        trip_df (pd.DataFrame): Trip data containing timestamps and locations.
        start_time_col (str): Column name for trip start timestamp.
        end_time_col (str): Column name for trip end timestamp.
        valid_duration_hour (float): Maximum allowed trip duration (in hours).
        start_location_col (str): Column name for trip start location ID.
        end_location_col (str): Column name for trip end location ID.
        trip_id_col (str, optional): Column name for unique trip ID. If None, index will be used.
        location_ids (iterable, optional): If provided, only retain records where location_id is in this list.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]:
            - Cleaned original trip_df with standardized time columns and trip_id
            - Influence DataFrame with columns ['trip_id', 'trip_time', 'location_id']
    """
    trip_df = trip_df.copy()
    trip_df[start_time_col] = pd.to_datetime(trip_df[start_time_col])
    trip_df[end_time_col] = pd.to_datetime(trip_df[end_time_col])

    # Ensure correct time ordering
    trip_df['start_datetime'] = trip_df[[start_time_col, end_time_col]].min(axis=1)
    trip_df['end_datetime'] = trip_df[[start_time_col, end_time_col]].max(axis=1)

    # Filter by trip duration
    trip_df['duration'] = (trip_df['end_datetime'] - trip_df['start_datetime']).dt.total_seconds() / 3600
    trip_df = trip_df[trip_df['duration'].between(0, valid_duration_hour)]

    # Ensure trip_id exists
    if trip_id_col is None:
        trip_df['trip_id'] = trip_df.index
        trip_id_col = 'trip_id'

    # Round timestamps to the hour
    trip_df['start_rounded_hour'] = trip_df['start_datetime'].dt.floor('h')
    trip_df['end_rounded_hour'] = trip_df['end_datetime'].dt.floor('h')

    # Prepare event frames
    pickup_df = trip_df[[trip_id_col, 'start_rounded_hour', start_location_col]].copy()
    pickup_df.columns = ['trip_id', 'trip_time', 'location_id']

    dropoff_df = trip_df[[trip_id_col, 'end_rounded_hour', end_location_col]].copy()
    dropoff_df.columns = ['trip_id', 'trip_time', 'location_id']

    # Add influence hours
    pickup_influence_df = pickup_df.copy()
    pickup_influence_df['trip_time'] -= pd.Timedelta(hours=1)

    dropoff_influence_df = dropoff_df.copy()
    dropoff_influence_df['trip_time'] += pd.Timedelta(hours=1)

    # Combine and deduplicate
    trip_with_influence_df = pd.concat([pickup_df, pickup_influence_df, dropoff_df, dropoff_influence_df], ignore_index=True)

    if location_ids is not None:
        trip_with_influence_df = trip_with_influence_df[trip_with_influence_df['location_id'].isin(location_ids)]

    trip_with_influence_df = trip_with_influence_df.drop_duplicates()

    return trip_df, trip_with_influence_df

### generate_hourly_passenger_influence_by_location

In [5]:
# Replaced with gen_valid_trip_influence_df which is more general, could be removed
def generate_hourly_passenger_influence_by_location(trip_data_filepath, location_ids=None):
    """
    Aggregates hourly passenger counts by location, accounting for temporal influence
    around each taxi trip (1 hour before pickup and 1 hour after dropoff).

    Assumptions:
        - Each trip influences the pickup location 1 hour before and during pickup.
        - Each trip influences the dropoff location during and 1 hour after dropoff.
        - Trips longer than 3 hours (top 0.03%) are considered outliers and will be excluded.
        - Missing passenger counts are imputed using the daily average per pickup location,
          falling back to the global average if necessary.
        - If `location_ids` is provided, only records with location_id in that list will be retained.

    Parameters:
        trip_data_filepath (str): Path to a Parquet file containing NYC yellow taxi trip records.
        location_ids (Iterable[int], optional): A list or set of location IDs to retain. 
                                                If None, all location_ids are included.

    Returns:
        pd.DataFrame: A DataFrame with columns ['trip_time', 'location_id', 'passenger_count'],
                      representing total influenced passenger counts per hour and location.
    """

    taxi_df = pd.read_parquet(trip_data_filepath, engine='pyarrow')

    taxi_df['pickup_datetime'] = np.minimum(taxi_df['tpep_pickup_datetime'], taxi_df['tpep_dropoff_datetime'])
    taxi_df['dropoff_datetime'] = np.maximum(taxi_df['tpep_pickup_datetime'], taxi_df['tpep_dropoff_datetime'])

    taxi_df['pickup_rounded_hour'] = taxi_df['pickup_datetime'].dt.floor('h')
    taxi_df['pickup_date'] = taxi_df['pickup_datetime'].dt.date
    taxi_df['pickup_hour'] = taxi_df['pickup_datetime'].dt.hour

    taxi_df['dropoff_rounded_hour'] = taxi_df['dropoff_datetime'].dt.floor('h')
    taxi_df['dropoff_date'] = taxi_df['dropoff_datetime'].dt.date
    taxi_df['dropoff_hour'] = taxi_df['dropoff_datetime'].dt.hour

    taxi_df['duration'] = (taxi_df['dropoff_datetime'] - taxi_df['pickup_datetime']).dt.total_seconds() / 3600
    taxi_df = taxi_df[taxi_df['duration'].between(0, 3)]

    group_avg = taxi_df.groupby(['PULocationID', 'pickup_date'])['passenger_count'].mean().reset_index(name='mean_cnt')
    taxi_df = taxi_df.merge(group_avg, on=['PULocationID', 'pickup_date'], how='left')

    global_avg = int(taxi_df['passenger_count'].mean())
    taxi_df['mean_cnt'] = taxi_df['mean_cnt'].fillna(global_avg)
    taxi_df['passenger_count'] = np.where(taxi_df['passenger_count'].notna(), taxi_df['passenger_count'], taxi_df['mean_cnt']).round().astype(int)

    taxi_df['trip_id'] = taxi_df.index

    pickup_df = taxi_df[['trip_id', 'pickup_rounded_hour', 'PULocationID']].copy()
    pickup_df.columns = ['trip_id', 'trip_time', 'location_id']
    dropoff_df = taxi_df[['trip_id', 'dropoff_rounded_hour', 'DOLocationID']].copy()
    dropoff_df.columns = ['trip_id', 'trip_time', 'location_id']

    pickup_influence_df = pickup_df.copy()
    pickup_influence_df['trip_time'] = pickup_influence_df['trip_time'] - pd.Timedelta(hours=1)
    dropoff_influence_df = dropoff_df.copy()
    dropoff_influence_df['trip_time'] = dropoff_influence_df['trip_time'] + pd.Timedelta(hours=1)

    trip_with_influence_df = pd.concat([pickup_df, pickup_influence_df, dropoff_df, dropoff_influence_df])

    # If location_ids are provided, filter the records
    if location_ids is not None:
        trip_with_influence_df = trip_with_influence_df[trip_with_influence_df['location_id'].isin(location_ids)]

    trip_with_influence_df = trip_with_influence_df.drop_duplicates()

    final_taxi_df = trip_with_influence_df.merge(taxi_df[['trip_id', 'passenger_count']], on='trip_id')
    final_taxi_df = final_taxi_df.groupby(['trip_time', 'location_id'])['passenger_count'].sum().reset_index()

    return final_taxi_df

# TLC Trip Record Data – Yellow Taxi Trip Records
- Source: [NYC TLC Yellow Taxi Trip Records](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
- Instructions:
  1. Download the monthly trip record datasets.
  2. Save all files in the `raw_data/` directory before running any parsing scripts.
- Date Range: January 2024 to March 2025
- The processed bike data has been saved to the [shared Google Drive folder](https://drive.google.com/drive/u/2/folders/1Yr7l0EcolHcL2TDrrq72ZUix2FNEc5X2)

In [6]:
station_filepath = '../manhattan_grid/taxi_location_grid_mapping.csv'
manhattan_grid = load_geoDataFrame(station_filepath)
manhattan_station_ids = manhattan_grid['location_id'].unique()

In [13]:
source_taxi_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [36]:
final_result_df = pd.DataFrame()
folder_path = 'raw_data/'

file_paths = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.parquet')])

for file in file_paths:
    print(f"Processing {file}...")
    source_taxi_df = pd.read_parquet(file, engine='pyarrow')

    # Generate valid trips and influence periods
    taxi_df, taxi_with_influence_df = gen_valid_trip_influence_df(
        trip_df=source_taxi_df,
        start_time_col='tpep_pickup_datetime',
        end_time_col='tpep_dropoff_datetime',
        valid_duration_hour=3,
        start_location_col='PULocationID',
        end_location_col='DOLocationID',
        trip_id_col=None,
        location_ids=manhattan_station_ids
    )

    # Compute daily average passenger count by PULocationID
    taxi_df['start_date'] = pd.to_datetime(taxi_df['start_datetime']).dt.date
    group_avg = taxi_df.groupby(['PULocationID', 'start_date'])['passenger_count'].mean().reset_index(name='mean_cnt')

    # Compute global average passenger count
    global_avg = int(taxi_df['passenger_count'].mean())

    # Merge mean into main df
    taxi_df = taxi_df.merge(group_avg, on=['PULocationID', 'start_date'], how='left')
    taxi_df['mean_cnt'] = taxi_df['mean_cnt'].fillna(global_avg)
    taxi_df['passenger_count'] = np.where(taxi_df['passenger_count'].notna(), taxi_df['passenger_count'], taxi_df['mean_cnt']).round().astype(int)

    # Aggregate by hour and location
    result_df = taxi_with_influence_df.merge(
        taxi_df[['trip_id', 'passenger_count']], on='trip_id'
    )
    result_df = result_df.groupby(['trip_time', 'location_id'])['passenger_count'].sum().reset_index()

    # Append result to final frame
    final_result_df = pd.concat([final_result_df, result_df], ignore_index=True)
final_result_df

Processing raw_data/yellow_tripdata_2024-01.parquet...
Processing raw_data/yellow_tripdata_2024-02.parquet...
Processing raw_data/yellow_tripdata_2024-03.parquet...
Processing raw_data/yellow_tripdata_2024-04.parquet...
Processing raw_data/yellow_tripdata_2024-05.parquet...
Processing raw_data/yellow_tripdata_2024-06.parquet...
Processing raw_data/yellow_tripdata_2024-07.parquet...
Processing raw_data/yellow_tripdata_2024-08.parquet...
Processing raw_data/yellow_tripdata_2024-09.parquet...
Processing raw_data/yellow_tripdata_2024-10.parquet...
Processing raw_data/yellow_tripdata_2024-11.parquet...
Processing raw_data/yellow_tripdata_2024-12.parquet...
Processing raw_data/yellow_tripdata_2025-01.parquet...
Processing raw_data/yellow_tripdata_2025-02.parquet...
Processing raw_data/yellow_tripdata_2025-03.parquet...


,trip_time,location_id,passenger_count
0,2002-12-31 21:00:00,170,2
1,2002-12-31 22:00:00,170,2
2,2002-12-31 23:00:00,170,2
3,2003-01-01 00:00:00,170,2
4,2009-01-01 22:00:00,137,1
...,...,...,...
700297,2025-04-01 01:00:00,246,4
700298,2025-04-01 01:00:00,249,10
700299,2025-04-01 01:00:00,261,3
700300,2025-04-01 01:00:00,262,9


In [37]:
# Re-aggregate passenger counts by hour and location
final_result_df = final_result_df.groupby(['trip_time', 'location_id'])['passenger_count'].sum().reset_index()

# Filter for date range: 2024-01-01 to 2025-03-31 (inclusive)
start_date = pd.to_datetime('2024-01-01')
end_date = pd.to_datetime('2025-04-01')  # exclusive upper bound
final_result_df = final_result_df[
    final_result_df['trip_time'].between(start_date, end_date - pd.Timedelta(seconds=1))
]

# Save to CSV
final_result_df.to_csv('yellow_taxi_20240101_20250331.csv', index=False)

In [39]:
final_result_df

,trip_time,location_id,passenger_count
124,2024-01-01 00:00:00,4,118
125,2024-01-01 00:00:00,12,11
126,2024-01-01 00:00:00,13,96
127,2024-01-01 00:00:00,24,84
128,2024-01-01 00:00:00,41,165
...,...,...,...
697046,2025-03-31 23:00:00,246,236
697047,2025-03-31 23:00:00,249,367
697048,2025-03-31 23:00:00,261,88
697049,2025-03-31 23:00:00,262,221
